# LegacyAgent V1
# Day 6 — Fine-Tuned Model Evaluation

Reusing the frozen Day 2 protocol exactly, no changes:

- Same 5-case held-out test set (frozen_test_benchmark.jsonl)
- Same normalization + WER scoring (copied verbatim from notebook 02)
- Same entity vocabulary (entity_terms.json)

Model under test: Qwen3-ASR-1.7B + LoRA adapter
(checkpoints/qlora_v1_run2_eosfix/final_adapter -- the EOS-fixed run, see notebook 05)

Important: don't touch the eval methodology in this notebook. If anything here drifts from notebook 02, the baseline-vs-finetuned comparison stops being valid.

In [1]:
# ============================================================
# DAY 6 — CELL 1
# Mount Drive and set paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/legacyagent")
EVAL_DIR = ROOT / "eval"
ADAPTER_DIR = ROOT / "checkpoints" / "qlora_v1_run2_eosfix" / "final_adapter"
RESULTS_DIR = ROOT / "results" / "day6_finetuned_eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert ADAPTER_DIR.exists(), f"Adapter not found: {ADAPTER_DIR}"
print("Adapter path:", ADAPTER_DIR)
print("Results will save to:", RESULTS_DIR)

Mounted at /content/drive
Adapter path: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2/final_adapter
Results will save to: /content/drive/MyDrive/legacyagent/results/day6_finetuned_eval


In [2]:
# ============================================================
# DAY 6 — CELL 2
# Install pinned dependencies — matches Day 5's working setup
# ============================================================

!pip -q uninstall -y peft transformers
!pip -q install \
    transformers==5.13.0 \
    accelerate \
    peft \
    bitsandbytes \
    librosa \
    soundfile \
    jiwer

import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
assert transformers.__version__ == "5.13.0", transformers.__version__
print("Versions locked correctly")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.6 MB/s eta 0:00:00
transformers: 5.13.0
peft: 0.20.0
Versions locked correctly


In [3]:
# ============================================================
# DAY 6 — CELL 3
# FROZEN normalization + WER (copied verbatim from
# notebook 02, Day 2 — Cell 4)
# ============================================================

import re
import unicodedata
from jiwer import wer, process_words

def normalize_for_wer(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[‐-‒–—−]+", " ", text)
    text = re.sub(r"[^\w\s']", " ", text)
    text = text.replace("'", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def calculate_corpus_wer(references, hypotheses):
    normalized_refs = [normalize_for_wer(t) for t in references]
    normalized_hyps = [normalize_for_wer(t) for t in hypotheses]
    return wer(normalized_refs, normalized_hyps)

# Same hard sanity check as Day 2 — must still pass here
assert calculate_corpus_wer(
    ["Hernandez v. Mesa", "Amicus curiae"],
    ["hernandez v mesa", "amicus curiae"]
) == 0.0
print("✓ Frozen normalizer matches Day 2 exactly")

✓ Frozen normalizer matches Day 2 exactly


In [4]:
# ============================================================
# DAY 6 — CELL 4
# Load the frozen Day 2 test benchmark — same 5 held-out cases
# ============================================================

import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_benchmark = load_jsonl(EVAL_DIR / "frozen_test_benchmark.jsonl")

with open(EVAL_DIR / "evaluation_protocol.json") as f:
    protocol = json.load(f)

print("Test segments:", len(test_benchmark))
print("Cases:", sorted(set(row["case_id"] for row in test_benchmark)))
print("Protocol frozen_at:", protocol.get("frozen_at", "n/a"))

Test segments: 1163
Cases: ['1996_96-318', '2000_99-1977', '2008_07-1015', '2013_13-115', '2016_15-118']
Protocol frozen_at: n/a


In [5]:
# ============================================================
# DAY 6 — CELL 5
# Load 4-bit base model + fine-tuned adapter
# Variable names match Day 2 exactly (qwen_model, qwen_processor)
# so the frozen transcribe function needs zero edits.
# ============================================================

import torch
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf + LoRA (qlora_v1_run2_eosfix/final_adapter)"
BASE_MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"

qwen_processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = Qwen3ASRForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

qwen_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
qwen_model.eval()

print("Fine-tuned model loaded as qwen_model / qwen_processor")

processor_config.json:   0%|          | 0.00/487 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.08GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Fine-tuned model loaded as qwen_model / qwen_processor


In [6]:
# ============================================================
# DAY 6 — CELL 6
# Frozen inference function — copied verbatim from
# notebook 02, Day 2 Cells 10 & 11 (parser + transcribe fn)
# Reverted: removed repetition_penalty / no_repeat_ngram_size —
# those were a band-aid for the old EOS-less adapter. The
# EOS-fixed adapter loaded in Cell 5 stops on its own without
# needing generation-time workarounds.
# ============================================================

import re
import torch

def parse_qwen_asr_output(raw_output: str) -> str:
    text = raw_output.strip()
    if "<asr_text>" in text:
        text = text.split("<asr_text>", 1)[1]
    text = re.sub(r"<\|.*?\|>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def transcribe_qwen_segment(row, waveform, sample_rate):
    start_sample = int(row["start"] * sample_rate)
    end_sample = int(row["end"] * sample_rate)
    audio_segment = waveform[:, start_sample:end_sample]
    audio_array = audio_segment.squeeze(0).numpy()

    conversation = [{"role": "user", "content": [{"type": "audio", "audio": audio_array}]}]

    inputs = qwen_processor.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    )

    fixed_inputs = {}
    for key, value in inputs.items():
        if isinstance(value, torch.Tensor):
            value = value.to(qwen_model.device)
            if value.is_floating_point():
                value = value.to(torch.bfloat16)
            fixed_inputs[key] = value
        else:
            fixed_inputs[key] = value

    with torch.inference_mode():
        generated_ids = qwen_model.generate(**fixed_inputs, max_new_tokens=256, do_sample=False)

    prompt_length = fixed_inputs["input_ids"].shape[1]
    generated_only = generated_ids[:, prompt_length:]
    raw_output = qwen_processor.batch_decode(generated_only, skip_special_tokens=True)[0].strip()
    transcript = parse_qwen_asr_output(raw_output)

    return raw_output, transcript

print("Frozen inference function loaded")

Frozen inference function loaded


In [7]:
# Sanity check: run the loaded adapter on one segment before the full pass
import torchaudio

test_row = test_benchmark[0]
waveform, sample_rate = torchaudio.load(test_row["audio_path"])
assert sample_rate == 16000

raw_output, transcript = transcribe_qwen_segment(test_row, waveform, sample_rate)
print("REF :", test_row["text"])
print("PRED:", transcript)
print("PRED LEN:", len(transcript.split()), " REF LEN:", len(test_row["text"].split()))

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


REF : We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.
PRED: We'll hear argument first this morning in Case 15-118, Hernandez v. Mesa. Mr. Hilliard.
PRED LEN: 14  REF LEN: 15


In [8]:
# ============================================================
# DAY 6 — CELL 7
# Full checkpointed fine-tuned inference — mirrors Day 2 Cell 13
# ============================================================

import json, time
import torchaudio

FINETUNED_RESULTS_PATH = RESULTS_DIR / "qwen3_asr_finetuned_predictions.jsonl"

def load_existing_predictions(path):
    if not path.exists():
        return [], set()
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    completed = {r["segment_id"] for r in rows if r.get("status") == "success"}
    return rows, completed

def save_prediction(path, result):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

existing_rows, completed_ids = load_existing_predictions(FINETUNED_RESULTS_PATH)
remaining_rows = [r for r in test_benchmark if r["segment_id"] not in completed_ids]

print("Total segments:", len(test_benchmark))
print("Already completed:", len(completed_ids))
print("Remaining:", len(remaining_rows))

cached_case_id = None
cached_waveform = None
cached_sample_rate = None
success_count = 0
failure_count = 0

for i, row in enumerate(remaining_rows, start=1):
    if row["case_id"] != cached_case_id:
        cached_waveform, cached_sample_rate = torchaudio.load(row["audio_path"])
        assert cached_sample_rate == 16000
        cached_case_id = row["case_id"]
        print(f"\nLoaded case: {cached_case_id}")

    t0 = time.time()
    try:
        raw_output, transcript = transcribe_qwen_segment(row, cached_waveform, cached_sample_rate)
        result = {
            "segment_id": row["segment_id"], "case_id": row["case_id"],
            "case_name": row["case_name"], "start": row["start"], "end": row["end"],
            "duration": row["duration"], "reference": row["text"], "prediction": transcript,
            "raw_output": raw_output, "inference_seconds": round(time.time() - t0, 3),
            "model": MODEL_ID, "status": "success",
        }
        success_count += 1
    except Exception as e:
        result = {
            "segment_id": row["segment_id"], "case_id": row["case_id"],
            "reference": row["text"], "error": str(e), "model": MODEL_ID, "status": "error",
        }
        failure_count += 1

    save_prediction(FINETUNED_RESULTS_PATH, result)
    if i % 10 == 0:
        print(f"Processed {i}/{len(remaining_rows)}  (success={success_count}, failed={failure_count})")

print("\nDone. Success:", success_count, "Failed:", failure_count)

Total segments: 1163
Already completed: 1056
Remaining: 107

Loaded case: 2008_07-1015
Processed 10/107  (success=10, failed=0)
Processed 20/107  (success=20, failed=0)
Processed 30/107  (success=30, failed=0)
Processed 40/107  (success=40, failed=0)
Processed 50/107  (success=50, failed=0)
Processed 60/107  (success=60, failed=0)
Processed 70/107  (success=70, failed=0)
Processed 80/107  (success=80, failed=0)
Processed 90/107  (success=90, failed=0)
Processed 100/107  (success=100, failed=0)

Done. Success: 107 Failed: 0


In [10]:
# ============================================================
# DAY 6 — CELL 8
# Official fine-tuned corpus WER — same computation shape as
# Day 2 Cells 15/24
# ============================================================

from jiwer import wer

results = [json.loads(l) for l in open(FINETUNED_RESULTS_PATH, encoding="utf-8") if l.strip()]
finetuned_by_id = {r["segment_id"]: r for r in results if r.get("status") == "success"}

references, hypotheses = [], []
for row in test_benchmark:
    seg_id = row["segment_id"]
    if seg_id not in finetuned_by_id:
        continue
    references.append(normalize_for_wer(row["text"]))
    hypotheses.append(normalize_for_wer(finetuned_by_id[seg_id]["prediction"]))

finetuned_wer = wer(references, hypotheses)
print("FINE-TUNED QWEN3-ASR — OFFICIAL RESULT")
print("=" * 60)
print("Evaluation segments:", len(references))
print("CORPUS WER (%)     :", round(finetuned_wer * 100, 2))

FINE-TUNED QWEN3-ASR — OFFICIAL RESULT
Evaluation segments: 1163
CORPUS WER (%)     : 7.36


In [11]:
# ============================================================
# DAY 6 — CELL 9
# Entity-span WER — same vocabulary and matching logic as Day 2
# Fixed: entity_terms.json stores the list under "terms", not "candidate_terms"
# Fixed: frozen normalizer function is normalize_for_wer, not normalize_text
# Fixed: predictions are stored under "prediction", not "hypothesis";
#        also filter to status == "success" like cell 8 does
# ============================================================

with open(EVAL_DIR / "entity_terms.json") as f:
    entity_terms_data = json.load(f)

candidate_terms = [t["term"].lower() for t in entity_terms_data["terms"]]
print("Entity vocabulary size:", len(candidate_terms))

def extract_entity_spans(text, terms):
    text_norm = normalize_for_wer(text)
    found = []
    for term in terms:
        term_norm = normalize_for_wer(term)
        if term_norm and term_norm in text_norm:
            found.append(term_norm)
    return found

entity_refs, entity_hyps = [], []
for row in test_benchmark:
    seg_id = row["segment_id"]
    if seg_id not in finetuned_by_id:
        continue
    reference_text = row["text"]
    prediction_text = finetuned_by_id[seg_id]["prediction"]

    ref_spans = extract_entity_spans(reference_text, candidate_terms)
    hyp_spans = extract_entity_spans(prediction_text, candidate_terms)
    if ref_spans:
        entity_refs.append(" ".join(ref_spans))
        entity_hyps.append(" ".join(hyp_spans) if hyp_spans else "")

entity_wer = wer(entity_refs, entity_hyps) if entity_refs else None
print("Fine-tuned entity-span WER:", round(entity_wer * 100, 2) if entity_wer is not None else "n/a", "%")
print("Segments with entity mentions:", len(entity_refs))

Entity vocabulary size: 24
Fine-tuned entity-span WER: 22.99 %
Segments with entity mentions: 163


In [12]:
# ============================================================
# DAY 6 — CELL 10
# Final V1 deliverable table — baseline vs fine-tuned
# Fixed: cell 8 defines the variable as finetuned_wer, not overall_wer
# ============================================================

# Pull your actual Day 2 baseline numbers here
baseline_overall_wer = 7.71   # from notebook 02
baseline_entity_wer = 30.67   # from notebook 02

comparison = {
    "model": "Qwen3-ASR-1.7B + QLoRA (qlora_v1_run2_eosfix/final_adapter)",
    "overall_wer_pct": round(finetuned_wer * 100, 2),
    "entity_span_wer_pct": round(entity_wer * 100, 2) if entity_wer else None,
    "baseline_overall_wer_pct": baseline_overall_wer,
    "baseline_entity_wer_pct": baseline_entity_wer,
    "overall_wer_relative_improvement_pct": round(
        (baseline_overall_wer - finetuned_wer * 100) / baseline_overall_wer * 100, 2
    ),
    "entity_wer_relative_improvement_pct": round(
        (baseline_entity_wer - entity_wer * 100) / baseline_entity_wer * 100, 2
    ) if entity_wer else None,
}

print(json.dumps(comparison, indent=2))

with open(RESULTS_DIR / "day6_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("\nSaved:", RESULTS_DIR / "day6_comparison.json")

{
  "model": "Qwen3-ASR-1.7B + QLoRA (step 350, eval_loss 0.6072)",
  "overall_wer_pct": 7.36,
  "entity_span_wer_pct": 22.99,
  "baseline_overall_wer_pct": 7.71,
  "baseline_entity_wer_pct": 30.67,
  "overall_wer_relative_improvement_pct": 4.54,
  "entity_wer_relative_improvement_pct": 25.03
}

Saved: /content/drive/MyDrive/legacyagent/results/day6_finetuned_eval/day6_comparison.json


In [13]:
# Sanity check: inspect a handful of actual predictions vs references
import random
sample = random.sample([r for r in results if r.get("status") == "success"], 5)
for r in sample:
    print("REF :", r["reference"])
    print("PRED:", r["prediction"])
    print("PRED LEN:", len(r["prediction"].split()), " REF LEN:", len(r["reference"].split()))
    print("-" * 60)

REF : --The government's position is if -- although we haven't briefed it in this case -- the government's position would be the one articulated by Justice Scalia. But we recognize that is not something that the Court has yet held and so I think--
PRED: The government's position is -- although we haven't briefed it in this case, the government's position would be the one articulated by Justice Scalia. But we recognize that is not something that the Court has yet held.
PRED LEN: 37  REF LEN: 43
------------------------------------------------------------
REF : Oh, I didn't understand the substantive test. I thought the substantive test for excessive was it is excessive only if a reasonable officer would have known it was too much force. I thought that was the substantive test. So what is the substantive test, if that isn't it?
PRED: Oh, I then didn't understand the substantive test. I thought the substantive test for excessive was it is excessive only if a reasonable officer would have 